# 3-2. **Word Embedding**

In [1]:
!pip install -q gensim

In [2]:
import re
import pprint
from lxml import etree
from gensim.models import Word2Vec

import nltk
from nltk import word_tokenize, sent_tokenize
nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

## Word2Vec

### Data Preprocessing

In [ ]:
!wget https://raw.githubusercontent.com/kimtwan/NLP_lecture/master/data/ted_en-20160408.zip
!unzip ted_en-20160408.zip

In [4]:
targetXML = open('ted_en-20160408.xml', 'r', encoding='UTF8')

# Getting contents of <content> tag from the xml file
target_text = etree.parse(targetXML)
parse_text = '\n'.join(target_text.xpath('//content/text()'))

# Removing 'Sound-effect labels' using regular expression (i.e. (Audio), (Laughter))
content_text = re.sub(r'\([^)]*\)', '', parse_text)

In [5]:
content_text[:1000]

"Here are two reasons companies fail: they only do more of the same, or they only do what's new.\nTo me the real, real solution to quality growth is figuring out the balance between two activities: exploration and exploitation. Both are necessary, but it can be too much of a good thing.\nConsider Facit. I'm actually old enough to remember them. Facit was a fantastic company. They were born deep in the Swedish forest, and they made the best mechanical calculators in the world. Everybody used them. And what did Facit do when the electronic calculator came along? They continued doing exactly the same. In six months, they went from maximum revenue ... and they were gone. Gone.\nTo me, the irony about the Facit story is hearing about the Facit engineers, who had bought cheap, small electronic calculators in Japan that they used to double-check their calculators.\n\nFacit did too much exploitation. But exploration can go wild, too.\nA few years back, I worked closely alongside a European bio

In [6]:
# Tokenizing the sentence to process it by using NLTK library
sent_text = sent_tokenize(content_text)

# Removing punctuations and changing all characters to lower case
normalized_text = []
for string in sent_text:
     tokens = re.sub(r'[^a-z0-9]+', ' ', string.lower())
     normalized_text.append(tokens)

# Tokenising each sentence to process individual word
sentences = [word_tokenize(sentence) for sentence in normalized_text]

# Prints only 10 (tokenized) sentences
print(sentences[:10])

[['here', 'are', 'two', 'reasons', 'companies', 'fail', 'they', 'only', 'do', 'more', 'of', 'the', 'same', 'or', 'they', 'only', 'do', 'what', 's', 'new'], ['to', 'me', 'the', 'real', 'real', 'solution', 'to', 'quality', 'growth', 'is', 'figuring', 'out', 'the', 'balance', 'between', 'two', 'activities', 'exploration', 'and', 'exploitation'], ['both', 'are', 'necessary', 'but', 'it', 'can', 'be', 'too', 'much', 'of', 'a', 'good', 'thing'], ['consider', 'facit'], ['i', 'm', 'actually', 'old', 'enough', 'to', 'remember', 'them'], ['facit', 'was', 'a', 'fantastic', 'company'], ['they', 'were', 'born', 'deep', 'in', 'the', 'swedish', 'forest', 'and', 'they', 'made', 'the', 'best', 'mechanical', 'calculators', 'in', 'the', 'world'], ['everybody', 'used', 'them'], ['and', 'what', 'did', 'facit', 'do', 'when', 'the', 'electronic', 'calculator', 'came', 'along'], ['they', 'continued', 'doing', 'exactly', 'the', 'same']]


### Word2Vec - Continuous Bag-Of-Words (CBOW)

In [7]:
wv_cbow_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [8]:
similar_words = wv_cbow_model.wv.most_similar('man')
pprint.pprint(similar_words)

[('woman', 0.848620593547821),
 ('guy', 0.8036758303642273),
 ('lady', 0.7891245484352112),
 ('girl', 0.7664448618888855),
 ('boy', 0.7621200680732727),
 ('soldier', 0.7257599234580994),
 ('gentleman', 0.70683354139328),
 ('poet', 0.6984544396400452),
 ('kid', 0.694191038608551),
 ('friend', 0.665217936038971)]


### Word2Vec - Skip Gram

In [9]:
wv_sg_model = Word2Vec(sentences=sentences, vector_size=100, window=5, min_count=5, workers=4, sg=1)

In [10]:
similar_words = wv_sg_model.wv.most_similar('man')
pprint.pprint(similar_words)

[('woman', 0.7768645882606506),
 ('guy', 0.7399013638496399),
 ('boy', 0.6923502683639526),
 ('gentleman', 0.6873252391815186),
 ('soldier', 0.6817033886909485),
 ('rabbi', 0.6672297716140747),
 ('comedian', 0.6597772836685181),
 ('lady', 0.6566938757896423),
 ('maid', 0.653414249420166),
 ('waitress', 0.6524035930633545)]


## Word2Vec vs FastText

Let's try to find out the difference between Word2Vec and FastText

Word2Vec - Skipgram cannot find similar word 'electrofishing' as 'electrofishing' is not in the vocabulary - so you can see the error

In [11]:
similar_words = wv_sg_model.wv.most_similar('electrofishing')
pprint.pprint(similar_words)

KeyError: "Key 'electrofishing' not present in vocabulary"

### FastText - Skip Gram

You can find that FastText works extremely well

In [12]:
from gensim.models import FastText

In [13]:
ft_sg_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4, sg=1)

In [14]:
result = ft_sg_model.wv.most_similar('electrofishing')
pprint.pprint(result)

[('electrolux', 0.8650190830230713),
 ('electrolyte', 0.8618406653404236),
 ('electroshock', 0.8538460731506348),
 ('electro', 0.8485544323921204),
 ('electroencephalogram', 0.8484968543052673),
 ('airbus', 0.830485999584198),
 ('electrochemical', 0.8299474716186523),
 ('electrogram', 0.8253276944160461),
 ('electric', 0.8239025473594666),
 ('airbag', 0.818535566329956)]


### FastText - Continuous Bag-Of-Words (CBOW)

In [15]:
ft_cbow_model = FastText(sentences, vector_size=100, window=5, min_count=5, workers=4, sg=0)

In [16]:
result = ft_cbow_model.wv.most_similar('electrofishing')
pprint.pprint(result)

[('fishing', 0.9174803495407104),
 ('flushing', 0.9060041308403015),
 ('flashing', 0.9041042327880859),
 ('smashing', 0.9005283713340759),
 ('licensing', 0.9004632234573364),
 ('vanishing', 0.898223340511322),
 ('flourishing', 0.8963751196861267),
 ('crashing', 0.8953514695167542),
 ('unleashing', 0.8943328261375427),
 ('refreshing', 0.893194317817688)]


## King + Woman - Man = ?

Try both CBOW and Skip Gram model to calculate 'King - Man + Woman = ?'

In [17]:
result = wv_cbow_model.wv.most_similar(positive=['king' , 'woman'], negative=['man'], topn=1)
print(result)

[('president', 0.7799447178840637)]


In [18]:
result = wv_sg_model.wv.most_similar(positive=['king' , 'woman'], negative=['man'], topn=1)
print(result)

[('queen', 0.6685914993286133)]


In [19]:
result = ft_cbow_model.wv.most_similar(positive=['king' , 'woman'], negative=['man'], topn=1)
print(result)

[('kidding', 0.8995832800865173)]


In [20]:
result = ft_sg_model.wv.most_similar(positive=['king' , 'woman'], negative=['man'], topn=1)
print(result)

[('jarring', 0.7334946990013123)]


This is not what we expected...Probably not enough data to answer as 'Queen'


# Play with Colab Form Fields
**The Form** supports multiple types of fields, including **input fields**, **dropdown menus**.

You can edit this section by double-clicking it.

Let's get familiar by changing the value in each input field (on the right) and checking the changes in the code (on the left) - vice versa

In [21]:
# @title Example form fields
# @markdown please put description

string = 'examples'  # @param {type: 'string'}
slider_value = 143  # @param {type: 'slider', min: 100, max: 200}
number = 10253  # @param {type: 'number'}
date = '2020-01-05'  # @param {type: 'date'}
pick_me = 'tuesday'  # @param ['monday', 'tuesday', 'wednesday', 'thursday']
select_or_input = 'apples' # @param ['apples', 'bananas', 'oranges'] {allow-input: true}


# print the output
print('string is',string)
print('slider_value',slider_value)

string is examples
slider_value 143


# Exercise
In this exercise, you need to implement a **'Word Algebra Calculator'  interface** using Word2Vec and FastText trained by the provided TED Scripts. The interface can be built by Colab Form Fields we just learned above.

What the users can do through the interface are:


1.   Input the word formula in the text field, e.g. King - Man + Woman
2.   Select the word embedding model from dropdown menu, either Word2Vec or FastText
3.   Select the training architecture from dropdown menu, either CBOW or Skip Gram
4.   Get(print out) the resulted word of the input formula by running the form (same to running the code section)



Note:
1. Please **do not** put the training process into your form section.
2. Please make your interface 'user-friendly' and instructional for users to use, e.g. by adding proper explaination or guide
3. We will use formula like 'Word1 + Word2 + Word3 - Word4' to test your interface, the number of the words and the sign between each two words can vary.



## 1.Build your word embedding models

In [ ]:
## Please generate four different types of word embedding models with TED data
## The parameter for all four models: vector_size=100, window=5, min_count=5, workers=4.



##2.Build your Interface

You can edit the following form elements to build your interface

In [ ]:
# @title Word Algebra Calculator

# @markdown Please select the model and formula to calculate the word algebra

# Get the input


# @markdown Now you can activate the Calculator by running this section

## 1.choose the corresponding model


## 2.processing the formula to extract the postive and negative word list


## 3.calculate the formula for an similar word using the selected model


## 4.print out the most similar word after
